# Levy + Neighbor Joint Fit

这个 notebook 用于拟合图步长联合模型，并和两个简化模型做对比：

- `joint`: Levy length + neighbor softmax + canonical effective colors
- `levy_only`: 只有 Levy length + canonical effective colors
- `neighbor_only`: 只有 neighbor softmax + canonical effective colors

对每个被试单独拟合参数，并以 `joint` 作为 baseline 比较 `NLL / AIC / BIC`。


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

from pathlib import Path
import sys
import importlib
from matplotlib import font_manager

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'fit_levy_neighbor_joint.py').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'model code' / 'fitting'
sys.path.insert(0, str(NOTEBOOK_DIR))

import fit_levy_neighbor_joint
importlib.reload(fit_levy_neighbor_joint)

from fit_levy_neighbor_joint import (
    build_joint_steps,
    get_global_graph_length_support,
    fit_all_participants_model_comparison,
)

plt.style.use('ggplot')

preferred_fonts = [
    'Source Han Sans SC',
    'Heiti SC',
    'PingFang SC',
    'STSong',
    'Arial Unicode MS',
]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
usable_fonts = [f for f in preferred_fonts if f in available_fonts]
if usable_fonts:
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = usable_fonts + ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)


In [ ]:
joint_steps = build_joint_steps(include_practice=False)
support = get_global_graph_length_support(joint_steps)

summary_df = pd.DataFrame([
    {
        'n_steps': len(joint_steps),
        'n_participants': joint_steps['participant'].nunique(),
        'n_rounds': joint_steps['round_label'].nunique(),
        'graph_support': tuple(int(x) for x in support),
    }
])
display(summary_df)
display(joint_steps.head())


In [ ]:
results_df = fit_all_participants_model_comparison(include_practice=False)
display(results_df.round(6))


## Parameter Estimates

下面只看每个被试在三种模型下的关键参数估计。

In [ ]:
param_table = results_df[['participant', 'model', 'b_hat', 'beta_hat', 'n_steps', 'll', 'nll', 'aic', 'bic']].copy()
display(param_table.round(6))

display(
    param_table.pivot(index='participant', columns='model', values='b_hat').round(6)
)
display(
    param_table.pivot(index='participant', columns='model', values='beta_hat').round(6)
)


## Compare Against Joint Baseline

这里把 `joint` 作为 baseline，因此：

- `delta_nll_vs_joint > 0` 表示比联合模型更差
- `delta_aic_vs_joint > 0` 表示比联合模型更差
- `delta_bic_vs_joint > 0` 表示比联合模型更差


In [ ]:
import numpy as np

compare_table = results_df[['participant', 'model', 'nll', 'aic', 'bic', 'delta_nll_vs_joint', 'delta_aic_vs_joint', 'delta_bic_vs_joint']].copy()
display(compare_table.round(6))

summary_compare = (
    compare_table.groupby('model', as_index=False)
    .agg(
        mean_nll=('nll', 'mean'),
        mean_aic=('aic', 'mean'),
        mean_bic=('bic', 'mean'),
        mean_delta_nll_vs_joint=('delta_nll_vs_joint', 'mean'),
        se_delta_nll_vs_joint=('delta_nll_vs_joint', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
        mean_delta_aic_vs_joint=('delta_aic_vs_joint', 'mean'),
        se_delta_aic_vs_joint=('delta_aic_vs_joint', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
        mean_delta_bic_vs_joint=('delta_bic_vs_joint', 'mean'),
        se_delta_bic_vs_joint=('delta_bic_vs_joint', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
    )
    .sort_values('mean_aic')
)
display(summary_compare.round(6))


In [ ]:
plot_order = ['levy_only', 'neighbor_only', 'joint']
plot_labels = {
    'levy_only': 'Levy only',
    'neighbor_only': 'Neighbor only',
    'joint': 'Levy + neighbor',
}
bar_colors = {
    'levy_only': '#c8c8c8',
    'neighbor_only': '#9db7e5',
    'joint': '#f3b07a',
}

plot_df = summary_compare.set_index('model').loc[plot_order].reset_index()
participant_order = sorted(compare_table['participant'].unique())
participant_colors = {
    participant: plt.cm.tab20(i % 20) for i, participant in enumerate(participant_order)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 10), sharex=True)

configs = [
    ('delta_aic_vs_joint', 'mean_delta_aic_vs_joint', 'se_delta_aic_vs_joint', 'AIC', 'ΔAIC (model - joint)'),
    ('delta_bic_vs_joint', 'mean_delta_bic_vs_joint', 'se_delta_bic_vs_joint', 'BIC', 'ΔBIC (model - joint)'),
    ('delta_nll_vs_joint', 'mean_delta_nll_vs_joint', 'se_delta_nll_vs_joint', 'NLL', 'ΔNLL (model - joint)'),
]

for ax, (raw_col, mean_col, se_col, title, ylabel) in zip(axes, configs):
    x = np.arange(len(plot_order))
    means = plot_df[mean_col].to_numpy(dtype=float)
    ses = plot_df[se_col].to_numpy(dtype=float)
    ax.bar(
        x,
        means,
        yerr=ses,
        capsize=5,
        width=0.68,
        color=[bar_colors[m] for m in plot_order],
        edgecolor='black',
        linewidth=0.8,
        alpha=0.8,
        zorder=1,
    )

    for j, model in enumerate(plot_order):
        sub = compare_table[compare_table['model'] == model].sort_values('participant')
        offsets = np.linspace(-0.12, 0.12, len(sub)) if len(sub) > 1 else np.array([0.0])
        for offset, (_, row) in zip(offsets, sub.iterrows()):
            ax.scatter(
                j + offset,
                row[raw_col],
                s=42,
                color=participant_colors[row['participant']],
                edgecolor='none',
                alpha=0.95,
                zorder=3,
                label=row['participant'] if (title == 'AIC' and model == plot_order[0]) else None,
            )

    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.8)
    ax.set_title(title, fontsize=20)
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels([plot_labels[m] for m in plot_order], rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Participant', loc='upper center', ncol=min(10, len(labels)), bbox_to_anchor=(0.5, 0.98))
fig.suptitle('Model differences relative to the joint model (bars = mean ± SE, dots = participants)', y=1.04, fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.86])


## LL Decomposition

这里查看联合模型和两个简化模型中，长度项、piece 项、颜色项各自贡献了多少。

In [ ]:
ll_parts = results_df[['participant', 'model', 'll_length', 'll_piece', 'll_color', 'll']].copy()
display(ll_parts.round(6))

display(
    ll_parts.groupby('model', as_index=False)
    .agg(
        mean_ll_length=('ll_length', 'mean'),
        mean_ll_piece=('ll_piece', 'mean'),
        mean_ll_color=('ll_color', 'mean'),
        mean_ll_total=('ll', 'mean'),
    )
    .round(6)
)
